In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from package.RankAMIP.logistic import run_logistic_regression
from package.RankAMIP.data_script import make_BT_design_matrix
from package.RankAMIP.logistic import LogisticAMIP
from package.RankAMIP.logistic import find_closest_matchups
from package.RankAMIP.logistic import isRankingRobust
from package.RankAMIP.data_script import *

### Load Data

In [9]:
import ssl
import urllib.request
import pandas as pd

# Disable SSL verification
ssl._create_default_https_context = ssl._create_unverified_context


# Import datasets from
df = pd.concat([pd.read_csv(f'https://raw.githubusercontent.com/JeffSackmann/tennis_atp/refs/heads/master/atp_matches_{i}.csv') for i in [2020, 2021, 2022, 2023, 2024]], ignore_index=True)
#df = pd.read_csv('https://raw.githubusercontent.com/JeffSackmann/tennis_atp/refs/heads/master/atp_matches_2020.csv')

We will examine games from the 2020 to the 2024 season (inclusive).

In [10]:
df_2024_rankings = pd.read_csv('https://raw.githubusercontent.com/JeffSackmann/tennis_atp/refs/heads/master/atp_rankings_current.csv')

In [11]:
# inspect the available splits
df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2020-8888,Atp Cup,Hard,24,A,20200106,300,104925,NaN,NaN,...,51.0,39.0,6.0,10.0,6.0,8.0,2.0,9055.0,1.0,9985.0
1,2020-8888,Atp Cup,Hard,24,A,20200106,299,105138,NaN,NaN,...,35.0,21.0,6.0,9.0,5.0,10.0,10.0,2335.0,34.0,1251.0
2,2020-8888,Atp Cup,Hard,24,A,20200106,298,104925,NaN,NaN,...,57.0,35.0,25.0,14.0,6.0,11.0,2.0,9055.0,5.0,5705.0
3,2020-8888,Atp Cup,Hard,24,A,20200106,297,105583,NaN,NaN,...,54.0,39.0,14.0,12.0,0.0,1.0,34.0,1251.0,17.0,1840.0
4,2020-8888,Atp Cup,Hard,24,A,20200106,296,104745,NaN,NaN,...,55.0,37.0,10.0,14.0,1.0,5.0,1.0,9985.0,18.0,1775.0


In [54]:
top_10_ranked_playerids = df_2024_rankings['player'].head(10)
#top_10_ranked_playerids

Filter for games between top-ranked players, where both the winner_id and loser_id are in top_10_ranked_playerids.

In [55]:
top_matchups = df[df['winner_id'].isin(top_10_ranked_playerids) & df['loser_id'].isin(top_10_ranked_playerids)]
top_matchups.shape

(278, 49)

Filter any singletons

In [56]:
# We will focus on the subset of players who have played at least 20 matches.
game_counts = top_matchups['winner_id'].value_counts() + top_matchups['loser_id'].value_counts()
valid_players = game_counts[game_counts > 20].index

# Filter rows where both winner and loser are in valid_players
top_matchups = top_matchups[top_matchups['winner_id'].isin(valid_players) & top_matchups['loser_id'].isin(valid_players)]

In [57]:
top_matchups.shape

(278, 49)

We will choose to assign player_A and player_B based on alphabetical ordering of name.

In [58]:
rawBT = top_matchups[['winner_name', 'loser_name', 'winner_rank', 'loser_rank']]
rawBT['player_A'] = rawBT.apply(lambda x: min(x['winner_name'], x['loser_name']), axis=1)
rawBT['player_B'] = rawBT.apply(lambda x: max(x['winner_name'], x['loser_name']), axis=1)
rawBT.head()

/var/folders/d0/9cwd6hsd1jx46dktk5_bn85c0000gn/T/ipykernel_22194/3541483567.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rawBT['player_A'] = rawBT.apply(lambda x: min(x['winner_name'], x['loser_name']), axis=1)
/var/folders/d0/9cwd6hsd1jx46dktk5_bn85c0000gn/T/ipykernel_22194/3541483567.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rawBT['player_B'] = rawBT.apply(lambda x: max(x['winner_name'], x['loser_name']), axis=1)


,winner_name,loser_name,winner_rank,loser_rank,player_A,player_B
2,Novak Djokovic,Daniil Medvedev,2.0,5.0,Daniil Medvedev,Novak Djokovic
74,Stefanos Tsitsipas,Alexander Zverev,6.0,7.0,Alexander Zverev,Stefanos Tsitsipas
282,Alexander Zverev,Andrey Rublev,7.0,16.0,Alexander Zverev,Andrey Rublev
459,Stefanos Tsitsipas,Hubert Hurkacz,6.0,29.0,Hubert Hurkacz,Stefanos Tsitsipas
498,Daniil Medvedev,Jannik Sinner,5.0,68.0,Daniil Medvedev,Jannik Sinner


In [59]:
rawBT['winner_player_A'] = (rawBT['player_A'] == rawBT['winner_name']).astype(int)
rawBT.head()

/var/folders/d0/9cwd6hsd1jx46dktk5_bn85c0000gn/T/ipykernel_22194/4291864355.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rawBT['winner_player_A'] = (rawBT['player_A'] == rawBT['winner_name']).astype(int)


,winner_name,loser_name,winner_rank,loser_rank,player_A,player_B,winner_player_A
2,Novak Djokovic,Daniil Medvedev,2.0,5.0,Daniil Medvedev,Novak Djokovic,0
74,Stefanos Tsitsipas,Alexander Zverev,6.0,7.0,Alexander Zverev,Stefanos Tsitsipas,0
282,Alexander Zverev,Andrey Rublev,7.0,16.0,Alexander Zverev,Andrey Rublev,1
459,Stefanos Tsitsipas,Hubert Hurkacz,6.0,29.0,Hubert Hurkacz,Stefanos Tsitsipas,0
498,Daniil Medvedev,Jannik Sinner,5.0,68.0,Daniil Medvedev,Jannik Sinner,1


In [60]:
# Count wins for each player
win_counts = rawBT['winner_name'].value_counts()
win_counts

Novak Djokovic        52
Daniil Medvedev       39
Jannik Sinner         36
Carlos Alcaraz        35
Alexander Zverev      31
Stefanos Tsitsipas    23
Andrey Rublev         21
Taylor Fritz          17
Holger Rune           13
Hubert Hurkacz        11
Name: winner_name, dtype: int64

Create winner_player_a column.

In [61]:
rawBT.head()

,winner_name,loser_name,winner_rank,loser_rank,player_A,player_B,winner_player_A
2,Novak Djokovic,Daniil Medvedev,2.0,5.0,Daniil Medvedev,Novak Djokovic,0
74,Stefanos Tsitsipas,Alexander Zverev,6.0,7.0,Alexander Zverev,Stefanos Tsitsipas,0
282,Alexander Zverev,Andrey Rublev,7.0,16.0,Alexander Zverev,Andrey Rublev,1
459,Stefanos Tsitsipas,Hubert Hurkacz,6.0,29.0,Hubert Hurkacz,Stefanos Tsitsipas,0
498,Daniil Medvedev,Jannik Sinner,5.0,68.0,Daniil Medvedev,Jannik Sinner,1


In [62]:
rawBT['winner_player_a'] = rawBT.apply(lambda x: 1 if x['winner_name'] == x['player_A'] else 0, axis=1)

/var/folders/d0/9cwd6hsd1jx46dktk5_bn85c0000gn/T/ipykernel_22194/1670736560.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rawBT['winner_player_a'] = rawBT.apply(lambda x: 1 if x['winner_name'] == x['player_A'] else 0, axis=1)


In [63]:
for_BT = rawBT[['player_A', 'player_B', 'winner_player_a']]
for_BT.head()

,player_A,player_B,winner_player_a
2,Daniil Medvedev,Novak Djokovic,0
74,Alexander Zverev,Stefanos Tsitsipas,0
282,Alexander Zverev,Andrey Rublev,1
459,Hubert Hurkacz,Stefanos Tsitsipas,0
498,Daniil Medvedev,Jannik Sinner,1


In [64]:
# make the BT design matrix.
X, y, player_to_id = make_BT_design_matrix(for_BT, weight_tie = False)
X.shape, y.shape

((278, 9), (278,))

In [65]:
id_to_player = {v: k for k, v in player_to_id.items()}
id_to_player

{0: 'Daniil Medvedev',
 1: 'Alexander Zverev',
 2: 'Hubert Hurkacz',
 3: 'Novak Djokovic',
 4: 'Andrey Rublev',
 5: 'Jannik Sinner',
 6: 'Carlos Alcaraz',
 7: 'Holger Rune',
 8: 'Stefanos Tsitsipas',
 9: 'Taylor Fritz'}

In [66]:
# compute BT scores.
model_full = run_logistic_regression(X, y)

# prepend model 0, the reference model, which has score 0.
bt_scores = np.insert(model_full.coef_[0], 0, 0)

In [67]:
bt_scores

array([ 0.        , -0.42353183, -0.96083941,  0.94954231, -0.73824069,
       -0.10269594,  0.50078946, -0.35780031, -0.71665757, -0.47647137])

In [68]:
# combine bt_scores with player names
bt_scores_with_names = {id_to_player[i]: score for i, score in enumerate(bt_scores)}
dict(sorted(bt_scores_with_names.items(), key=lambda x: x[1], reverse=True))


{'Novak Djokovic': 0.949542310107934,
 'Carlos Alcaraz': 0.5007894629348154,
 'Daniil Medvedev': 0.0,
 'Jannik Sinner': -0.10269593890908249,
 'Holger Rune': -0.3578003110402152,
 'Alexander Zverev': -0.42353182736348366,
 'Taylor Fritz': -0.4764713716219253,
 'Stefanos Tsitsipas': -0.7166575737912605,
 'Andrey Rublev': -0.7382406945737989,
 'Hubert Hurkacz': -0.9608394106400994}

In [69]:
# Count wins for each player
win_counts = rawBT['winner_name'].value_counts()
win_counts

Novak Djokovic        52
Daniil Medvedev       39
Jannik Sinner         36
Carlos Alcaraz        35
Alexander Zverev      31
Stefanos Tsitsipas    23
Andrey Rublev         21
Taylor Fritz          17
Holger Rune           13
Hubert Hurkacz        11
Name: winner_name, dtype: int64

#### Run Top-k Robustness Check.

In [70]:
ks = [1, 5]
results = {}
for k in ks:
    alphaN = 1
    chatbotA = -1
    while chatbotA == -1:
        chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices = isRankingRobust(k, alphaN, X, y, weighted = False)
        results[(k, alphaN)] = (chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices)
        alphaN += 1

In [71]:
6 / len(X)

0.02158273381294964

In [72]:
# find the (k, alpha N) pairs that are non-robust.
results_nonrobust = {k: v for k, v in results.items() if v[0] != -1}
results_nonrobust

{(1, 6): (2,
  5,
  0.4487528471731186,
  -0.007501636060074257,
  array([236, 168, 251, 177, 202, 122])),
 (5, 1): (6, 0, 0.06573151632326846, -0.04476949541293118, array([120]))}

Restricting the arena to the top 10 players, we only need to drop 3 out of 67 matches.

In [73]:
from package.RankAMIP.plot_util import *
rankings = return_rankings_list(X, y, results, 1, 6, player_to_id)

In [74]:
# plot the rankings on the original arena
filename_to_save = 'fig/tennisTop30.png'
plot_title = 'Player Rankings in Tennis'
plot_bt_scores(X, y, rankings, alphaN, 10, plot_title, filename_to_save)

Note: The robustness of an analysis has a dependence on the size of the subset of data we choose. When we analyze the top-30 players (rather than top-10), we only need to drop 0.00453 of preferences to change the top-ranked model. Hypothesis is that larger 